# Classification — Early Settlement: SUN vs SOL

---

## Project Overview

**Objective:** Binary classification to distinguish between two types of early settlement:
- `SAN (1)` — client settled the contract early on their own initiative
- `SOL (0)` — client settled early due to refinancing or other external reason

**Data:** Aggregated dataset — 1 row per client, merged from 4 source tables:
`BDOSSTOTAL` · `CRC` · `CredScore` · `FAMA_LIGHT_`

---

## Notebook Structure

```
┌─────────────────────────────────────────────────────────────┐
│  0. SET UP                                                  │
│     Imports, constants, paths                               │
├─────────────────────────────────────────────────────────────┤
│  1. LOAD DATASET                                            │
│     Load aggregated client dataset (1 row per client)       │
├─────────────────────────────────────────────────────────────┤
│  2. TRAIN / TEST SPLIT                                      │
│     Split once — X_test goes in the drawer until Section 8  │
│     Define CV strategy (StratifiedKFold, k=5)               │
├─────────────────────────────────────────────────────────────┤
│  3. PREPROCESSING PIPELINE                                  │
│     ClientDataCleaner                                       │
│       → ClientOutlierHandler                                │
│         → ClientImputer                                     │
│           → ClientFeatureEngineer                           │
│             → ColumnTransformer  (encoding + scaling)[TODO] │
├─────────────────────────────────────────────────────────────┤
│  4. FIT & TRANSFORM                                         │
│     fit_transform(X_train)  ← learns from train only        │
│     transform(X_test)       ← applies, never learns         │
├─────────────────────────────────────────────────────────────┤
│  5. FEATURE SELECTION                  [TODO]               │
│     Computed on X_train_df only                             │
│     Same mask applied to X_test_df                          │
├─────────────────────────────────────────────────────────────┤
│  6. MODEL COMPARISON                   [TODO]               │
│     cross_validate on X_train_fs                            │
│     Compare baseline: LR · RF · GBM                         │
├─────────────────────────────────────────────────────────────┤
│  7. HYPERPARAMETER TUNING              [TODO]               │
│     RandomizedSearchCV on best model                        │
│     Still only uses X_train_fs                              │
├─────────────────────────────────────────────────────────────┤
│  8. FINAL EVALUATION  ← runs ONCE at the very end           │
│     predict(X_test_fs)                                      │
│     ROC-AUC · Classification Report · Confusion Matrix      │
└─────────────────────────────────────────────────────────────┘
```

---

## Anti-Leakage Rules

| Rule | Where enforced |
|---|---|
| `X_test` never influences any decision | Split in Section 2, only used in Section 8 |
| All statistics learned from train fold only | `fit_transform` only on `X_train` in Section 4 |
| Feature selection criteria computed on train only | Section 5 uses `X_train_df` exclusively |
| CV uses only `X_train_fs` | Sections 6 and 7 never touch `X_test_fs` |


## 0. SET UP

In [3]:
import src.code.io_utils as io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, roc_auc_score, classification_report

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline

from src.code.class_pipeline_functions import ClientDataCleaner, ClientOutlierHandler, ClientImputer, \
    ClientFeatureEngineer

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)

CLIENT_PATH = "../data/prepared/abt.parquet"

## 1. Load Aggregation Dataset

In [4]:
client_data = io.load(CLIENT_PATH)
client_data.head(3)

[LOAD] ../data/prepared/abt.parquet | shape: (148729, 87)


,CONTRIB,N_CONTRACTS,FIRST_DCREAT,LAST_DCREAT,LAST_DPOS,LAST_DATFIN,FIRST_D1FIN,MIN_DURDEG,MAX_DURDEG,MEDIAN_DURDEG,MIN_RANGPRO,MAX_RANGPRO,MEDIAN_RANGPRO,MIN_RANGCLI,MAX_RANGCLI,MEDIAN_RANGCLI,TOTAL_MTFIN,TOTAL_MTFINO,TOTAL_MENSALIDADE,TOTAL_CRD,TOTAL_SREC,TOTAL_RN,TOTAL_RD,LAST_RISK,MAX_RISKA,MIN_RESSO,MAX_RESSO,MEDIAN_RESSO,CSP,NBENF,EVER_SOL,N_SOL,EVER_SAN,N_SAN,EVER_RBT,N_RBT,LAST_OBS_DATE_SOL,LAST_OBS_DATE_SAN,LAST_OBS_DATE_RBT,IS_EARLY_SETTLER,IS_CHURN,MT_MENSAL_MIN,MT_MENSAL_MAX,MT_MENSAL_MEDIAN,COUNT_CL_MIN,COUNT_CL_MAX,COUNT_CL_MEDIAN,COUNT_AUTO_MIN,COUNT_AUTO_MAX,COUNT_AUTO_MEDIAN,COUNT_TOTAL_MIN,COUNT_TOTAL_MAX,COUNT_TOTAL_MEDIAN,MONTVENC_CL_MIN,MONTVENC_CL_MAX,MONTVENC_CL_MEDIAN,MONTVENC_CP_MIN,MONTVENC_CP_MAX,MONTVENC_CP_MEDIAN,MONTVENC_AUTO_MIN,MONTVENC_AUTO_MAX,MONTVENC_AUTO_MEDIAN,MONTVENC_HT_MIN,MONTVENC_HT_MAX,MONTVENC_HT_MEDIAN,DIVIDAS_CL_MIN,DIVIDAS_CL_MAX,DIVIDAS_CL_MEDIAN,DIVIDAS_CP_MIN,DIVIDAS_CP_MAX,DIVIDAS_CP_MEDIAN,DIVIDAS_AUTO_MIN,DIVIDAS_AUTO_MAX,DIVIDAS_AUTO_MEDIAN,DIVIDAS_HT_MIN,DIVIDAS_HT_MAX,DIVIDAS_HT_MEDIAN,kp_sqe_enc,ks_score_tier,ALLBD_N_Dossiers__N,ALLBD_A_CL__N,ALLBD_A_CP__N,ALLBD_IDADE_MEAN__N,ALLBD_N_events__N,sdem_SITFAM,sdem_HABITAT,sdem_age
0,00008246f87bcc3c17b90629bb183fe2e58795176310f0...,2,2024-06-25,2024-09-13,2025-11-17,2024-09-30,2024-06-27,84.0,120.0,102.0,3.0,13.0,8.0,19.0,19.0,19.0,24000.00,24000.00,438.761355,0.000,0.0,0.0,0.0,0,0.0,1513.466,1513.466,1513.466,80.0,0.0,0,0,1,2,0,0,NaT,2025-11-30,NaT,1,1,0.00,0.00,0.00,0.0,1.0,0.0,0.0,6.0,1.0,5.0,18.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,8000.00,0.00,1.65,3701.73,978.25,0.0,58633.86,15934.05,0.0,0.0,0.0,7.0,2.0,10.0,1.0,2.0,-438.9,10.0,S,F,33.0
1,0000ab2116257783438c70ff85a3e98f2d4194ebe53434...,1,2018-03-29,2018-03-29,2018-04-16,2018-04-16,2018-04-16,120.0,120.0,120.0,91.0,91.0,91.0,91.0,91.0,91.0,20000.00,20000.00,347.447280,8115.247,0.0,0.0,0.0,0,0.0,1113.258,1113.258,1113.258,80.0,1.0,0,0,0,0,0,0,NaT,NaT,NaT,0,0,538.67,547.45,545.07,4.0,12.0,4.0,0.0,0.0,0.0,10.0,30.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20308.94,69754.56,24983.22,5499.88,21274.83,7115.46,0.0,0.00,0.00,0.0,0.0,0.0,3.0,1.0,10.0,1.0,0.0,1166.9,10.0,C,P,52.0
2,0000c74654405ec1da4dbdcd00b86e397954043965d98e...,2,2001-09-21,2001-09-27,2014-06-27,2001-10-04,2001-09-21,60.0,60.0,60.0,21.0,21.0,21.0,21.0,21.0,21.0,19453.11,19453.11,608.913796,0.000,0.0,2.0,24.0,-9223372036854775808,0.0,678.483,678.483,678.483,60.0,2.0,1,2,0,0,0,0,2024-03-31,NaT,NaT,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN


## 2. Train/Test Split

In [5]:
X = client_data.drop(columns=['IS_EARLY_SETTLER'])
y = client_data['IS_EARLY_SETTLER']


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,                  # maintain the portion of SAN/SOL
    random_state=RANDOM_STATE,
)

print(f'\nTrain : {X_train.shape} — SAN rate: {y_train.mean():.2%}')
print(f'Test  : {X_test.shape}  — SAN rate: {y_test.mean():.2%}')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


Train : (118983, 86) — SAN rate: 27.82%
Test  : (29746, 86)  — SAN rate: 27.82%


## 3. Build Preprocessing Pipeline

No model here, this pipeline only preprocesses the data.
The model is added later after feature selection.

In [6]:
preprocessing_pipe = Pipeline([
    ('cleaning',  ClientDataCleaner( ##probably not necessary
                     datetime_cols  = ['DPOS', 'DCREAT', 'DATFIN', 'D1FIN'],
                     cols_to_remove = [],
                     verbose        = True,
                 )),
    ('outliers', ClientOutlierHandler(
                     methods   = ('iqr', 'mod_z'),
                     min_votes = 2,
                     iqr_k     = 1.5,
                     verbose   = True,
                 )),
    ('imputer',  ClientImputer(
                     numeric_strategy  = 'median',
                     datetime_strategy = 'ffill', ## not sure if this makes sense, but we don't want to drop date cols before feature engineering
                     verbose           = True,
                 )),
    ('fe',       ClientFeatureEngineer(
                     include_composite = True,
                     drop_date_cols    = True,
                     verbose           = True,
                 )),
    ## TODO ('ct',       ct),
])

print('Pipeline built successfully')
print(preprocessing_pipe)

Pipeline built successfully
Pipeline(steps=[('cleaning',
                 ClientDataCleaner(cols_to_remove=[],
                                   datetime_cols=['DPOS', 'DCREAT', 'DATFIN',
                                                  'D1FIN'],
                                   verbose=True)),
                ('outliers', ClientOutlierHandler(verbose=True)),
                ('imputer', ClientImputer(verbose=True)),
                ('fe', ClientFeatureEngineer(verbose=True))])


## 4. Fit & Transform

- `fit_transform` on `X_train`: pipeline learns all statistics from training data only
- `transform` on `X_test`: applies without learning anything new

**Never call `fit_transform` on `X_test`.**

In [7]:
# Fit on train, transform both
X_train_processed = preprocessing_pipe.fit_transform(X_train, y_train)
X_test_processed  = preprocessing_pipe.transform(X_test)

# Get feature names from the ColumnTransformer
feature_names = preprocessing_pipe.named_steps['ct'].get_feature_names_out()

# Convert to DataFrame for easier inspection and feature selection
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df  = pd.DataFrame(X_test_processed,  columns=feature_names)

print(f'X_train processed : {X_train_df.shape}')
print(f'X_test  processed : {X_test_df.shape}')
print(f'Any NaNs remaining: {X_train_df.isna().sum().sum()}') ##just to check

[ClientDataCleaner]
  Output shape     : (118983, 86)
  Columns removed  : —
  Dates converted  : —

[ClientOutlierHandler]
  Methods      : ('iqr', 'mod_z')  (min_votes=2)
  IQR k        : 1.5  |  Z threshold : 3.5
  Total flagged: 142390 cell(s) → set to NaN

    DIVIDAS_CP_MIN                      12857 outlier(s)
    DIVIDAS_CP_MAX                      9991 outlier(s)
    DIVIDAS_CP_MEDIAN                   9562 outlier(s)
    DIVIDAS_CL_MAX                      9200 outlier(s)
    DIVIDAS_AUTO_MAX                    8440 outlier(s)
    DIVIDAS_CL_MIN                      8161 outlier(s)
    TOTAL_CRD                           8076 outlier(s)
    TOTAL_MENSALIDADE                   7854 outlier(s)
    COUNT_CL_MAX                        7328 outlier(s)
    TOTAL_MTFINO                        7318 outlier(s)
    TOTAL_MTFIN                         7318 outlier(s)
    MAX_RESSO                           6406 outlier(s)
    MIN_RESSO                           6380 outlier(s)
    MEDIA

ValueError: Imputation incomplete: 78 NaN(s) remain in columns: ['LAST_OBS_DATE_SOL', 'LAST_OBS_DATE_RBT']. This may happen if a column type changed between fit() and transform().

## 5. Feature Selection

Feature selection is applied on `X_train_df` only.
The same selected columns are then applied to `X_test_df`.

**No leakage here because:**
- All selection criteria (variance, correlation, importance) are computed on `X_train_df`
- `X_test_df` is only filtered at the end using the mask learned from train

In [ ]:
## TODO


# Placeholder — keeps all features until you decide
selected_features = feature_names.tolist()

X_train_fs = X_train_df[selected_features]
X_test_fs  = X_test_df[selected_features]

print(f'Features after selection: {len(selected_features)}')

## 6. Model Comparison

Quick baseline comparison across models using cross-validation on `X_train_fs`.
Pick the best baseline before investing time in hyperparameter tuning.


## 7. Hyperparameter Tuning

Run `RandomizedSearchCV` on the best model from Section 6.
Still using only `X_train_fs`. `X_test_fs` remains untouched.

## 8. Final Evaluation on Test Set

**This section runs only once at the very end.**
The test set was never used for any decision up to this point.

In [ ]:
final_model = search.best_estimator_ ##it comes from the randomized search

y_pred      = final_model.predict(X_test_fs)
y_pred_prob = final_model.predict_proba(X_test_fs)[:, 1]

print('=' * 50)
print('FINAL TEST SET RESULTS')
print('=' * 50)
print(f'ROC-AUC  : {roc_auc_score(y_test, y_pred_prob):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['SOL', 'SUN']))

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['SOL', 'SUN'],
    ax=ax,
)
plt.title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.show()